In [2]:
import colorsys
from rich import print
_colors = ("#ffe119", "#a8f678", "#39c5c5", "#c5c19b", '#44ff00', "#df5e3e", "#f08a5d", '#00ffe1', '#ff0000')

In [23]:
def toHtml(r, g, b):
    return f'#{r:02x}{g:02x}{b:02x}'

def toRgb(html: str) -> tuple:
    hex_color = html.lstrip('#')
    rgb = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return rgb


def furthest_colors(html):
    rtn = []
    comp = complementary(html)
    rtn.append(comp)
    rtn += triadic(html)
    rtn += triadic(comp)
    return rtn


def generate_colors(amt, s=.75, v=1, offset=0):
    """ Generate `amt` number of colors evenly spaced around the color wheel
        with a given saturation and value
    """
    amt += 1
    return [toHtml(*map(lambda c: round(c*255), colorsys.hsv_to_rgb(*((offset + ((1/amt) * (i + 1))) % 1.001, s, v)))) for i in range(amt-1)]


def furthest_colors(html, amt=5, v_bias=0, s_bias=0):
    """ Gets the `amt` number of colors evenly spaced around the color wheel from the given color
        `v_bias` and `s_bias` are between 0-1 and offset the colors
    """
    amt += 1
    h, s, v = colorsys.rgb_to_hsv(*map(lambda c: c/255, toRgb(html)))

    return [toHtml(*map(lambda c: round(c*255), colorsys.hsv_to_rgb(*((h + ((1/amt) * (i + 1))) % 1.001, (s+s_bias) % 1.001, (v+v_bias) % 1.001)))) for i in range(amt-1)]


In [24]:
foreground_s = .75
foreground_v = 1
background_v_bias = .5
background_s_bias = .9

for c in generate_colors(8, s=foreground_s, v=foreground_v, offset=0/16):
    print(f'[{c}]{c}-----------------------------------------[/]')
    for opp in furthest_colors(c, amt=6, v_bias=background_v_bias, s_bias=background_v_bias):
        print(f'[{c} on {opp}]{c} on {opp}[/]  [{opp}]{opp}[/]')

#ffbf40-----------------------------------------

#ffbf40 on #6f7f60  #6f7f60

#ffbf40 on #607f6c  #607f6c

#ffbf40 on #60787f  #60787f

#ffbf40 on #63607f  #63607f

...

#ff40bf on #7e7f60  #7e7f60

#ff40bf on #637f60  #637f60

#ff40bf on #607f78  #607f78

#ff40bf on #606c7f  #606c7f

#ff40bf on #6f607f  #6f607f

In [50]:
# V2
from __future__ import annotations

import colorsys
import math


def _srgb_to_linear(c: float) -> float:
    return c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4


def _linear_to_srgb(c: float) -> float:
    return 12.92 * c if c <= 0.0031308 else 1.055 * c ** (1 / 2.4) - 0.055


def _rgb_to_lab(rgb: tuple[int, int, int]) -> tuple[float, float, float]:
    r, g, b = (_srgb_to_linear(x / 255) for x in rgb)

    # sRGB D65 -> XYZ
    x = r * 0.4124564 + g * 0.3575761 + b * 0.1804375
    y = r * 0.2126729 + g * 0.7151522 + b * 0.0721750
    z = r * 0.0193339 + g * 0.1191920 + b * 0.9503041

    # D65 reference white
    x /= 0.95047
    y /= 1.00000
    z /= 1.08883

    def f(t: float) -> float:
        return t ** (1 / 3) if t > 0.008856 else 7.787 * t + 16 / 116

    x, y, z = f(x), f(y), f(z)

    return (
        116 * y - 16,
        500 * (x - y),
        200 * (y - z),
    )


def _lab_to_rgb(lab: tuple[float, float, float]) -> tuple[int, int, int] | None:
    L, a, b = lab

    fy = (L + 16) / 116
    fx = a / 500 + fy
    fz = fy - b / 200

    def finv(t: float) -> float:
        t3 = t ** 3
        return t3 if t3 > 0.008856 else (t - 16 / 116) / 7.787

    x = 0.95047 * finv(fx)
    y = 1.00000 * finv(fy)
    z = 1.08883 * finv(fz)

    # XYZ -> linear sRGB
    r = 3.2404542 * x - 1.5371385 * y - 0.4985314 * z
    g = -0.9692660 * x + 1.8760108 * y + 0.0415560 * z
    b = 0.0556434 * x - 0.2040259 * y + 1.0572252 * z

    # Out of sRGB gamut
    if not (0 <= r <= 1 and 0 <= g <= 1 and 0 <= b <= 1):
        return None

    return tuple(
        round(max(0, min(1, _linear_to_srgb(x))) * 255)
        for x in (r, g, b)
    )


def _relative_luminance(rgb: tuple[int, int, int]) -> float:
    linear = [_srgb_to_linear(x / 255) for x in rgb]
    return (
        0.2126 * linear[0]
        + 0.7152 * linear[1]
        + 0.0722 * linear[2]
    )


def _contrast_ratio(
    a: tuple[int, int, int],
    b: tuple[int, int, int],
) -> float:
    l1 = _relative_luminance(a)
    l2 = _relative_luminance(b)
    return (max(l1, l2) + 0.05) / (min(l1, l2) + 0.05)


def generate_colors(amt: int, base: tuple, readability_distinctness_ratio=4.5) -> list[tuple]:
    if amt <= 0:
        return []

    if len(base) != 3 or any(not 0 <= x <= 255 for x in base):
        raise ValueError("base must be an RGB tuple with values from 0 to 255")

    base = tuple(map(int, base))
    base_lab = _rgb_to_lab(base)

    # Generate candidate colors.
    #
    # We vary lightness, hue, and chroma. Higher chroma gives more vivid
    # colors, while multiple lightness levels prevent everything from
    # collapsing into the same perceptual region.
    candidates: list[tuple[tuple[int, int, int], tuple[float, float, float]]] = []

    for L in range(20, 91, 5):
        for hue in range(0, 360, 5):
            for chroma in range(30, 101, 10):
                angle = math.radians(hue)

                lab = (
                    L,
                    chroma * math.cos(angle),
                    chroma * math.sin(angle),
                )

                rgb = _lab_to_rgb(lab)
                if rgb is None:
                    continue

                # WCAG AA-ish readability threshold.
                if _contrast_ratio(rgb, base) < readability_distinctness_ratio:
                    continue

                candidates.append((rgb, lab))

    if not candidates:
        raise ValueError("Could not find any readable colors for this base")

    # Pick the first color furthest from the base.
    first = max(
        candidates,
        key=lambda x: (
            (x[1][0] - base_lab[0]) ** 2
            + (x[1][1] - base_lab[1]) ** 2
            + (x[1][2] - base_lab[2]) ** 2
        ),
    )

    selected = [first]
    remaining = [c for c in candidates if c != first]

    # Farthest-point sampling:
    # At every step, choose the candidate whose nearest selected color
    # is as far away as possible.
    while len(selected) < amt:
        if not remaining:
            break

        best = max(
            remaining,
            key=lambda candidate: min(
                (
                    (candidate[1][0] - chosen[1][0]) ** 2
                    + (candidate[1][1] - chosen[1][1]) ** 2
                    + (candidate[1][2] - chosen[1][2]) ** 2
                )
                for chosen in selected
            ),
        )

        selected.append(best)
        remaining.remove(best)

    return [rgb for rgb, _ in selected]

In [51]:
base = (175, 211, 204)
htmlBase = toHtml(*base)
for c in generate_colors(18, base):
    # print(f'[{c} on {opp}]{c} on {opp}[/]  [{opp}]{opp}[/]')

    print(f'[{toHtml(*c)} on {htmlBase}]{toHtml(*c)} on {htmlBase}[/]')
    # console.print(c, style=c)
    # print(f'[{c}]{c}-----------------------------------------[/]')
    # for opp in furthest_colors(c, amt=6, v_bias=background_v_bias, s_bias=background_v_bias):
    # print(f'[{c} on {htmlBase}]{c} on {htmlBase}[/]  ')

#1417bd on #afd3cc

#345d02 on #afd3cc

#a40745 on #afd3cc

#0a5781 on #afd3cc

#4c2707 on #afd3cc

...

#7728ae on #afd3cc

#674f0b on #afd3cc

#9a156c on #afd3cc

#291b87 on #afd3cc

#863c2b on #afd3cc